# Predicting the Popularity of Online News

**Author:** Habib Kazemi  
**Course:** Big Data Analytics and Text Mining + 3CFU Project Work  
**Professor:** Dr. Stefano Lodi  
**Academic Year:** 2024-2025

---

## Classification Algorithms Implemented

This notebook implements **3 out of 6** classification algorithms using PySpark MLlib:

1. **Logistic Regression**
2. **Naive Bayes**  
3. **Decision Tree Classifier**

> **Note:** The remaining 3 algorithms are implemented in the companion notebook: `big_data_3cfu.ipynb`

## Methodology

This notebook uses **PySpark MLlib** to predict online news article popularity using a **binary classification approach** with **rolling window time-series validation**, following the methodology from the reference paper.

# Data Sources and References

## Dataset
**Online News Popularity Data Set** - [UCI Machine Learning Repository](https://archive.ics.uci.edu/ml/datasets/Online+News+Popularity)

##  Main Reference Paper

**Fernandes, K., Vinagre, P., & Cortez, P. (2015).** *A Proactive Intelligent Decision Support System for Predicting the Popularity of Online News.* In Portuguese Conference on Artificial Intelligence (EPIA 2015). Springer.

🔗 **Available at:** [Semantic Scholar](https://api.semanticscholar.org/CorpusID:7477506)


# Introduction

## Methodology Overview

This project follows the **rolling window time-series validation** approach as described in the reference paper to ensure robust model evaluation while respecting temporal dependencies in the data.

## Machine Learning Pipeline

I implemented **6 different classification algorithms** using **PySpark MLlib**, distributed across two notebooks:

### **Current Notebook (6CFU):**
4. **Random Forest Classifier**
5. **Gradient-Boosted Tree Classifier**
6. **Support Vector Machine (SVM) Classifier**

### **Companion Notebook (3CFU):**
1. **Logistic Regression**
2. **Naive Bayes**
3. **Decision Tree Classifier**


## Evaluation Metrics

All models are evaluated using comprehensive performance metrics:
- **ROC Curves** and **AUC (Area Under Curve)**
- **Accuracy**
- **F1-Score**
- **Precision** 
- **Recall**

This multi-metric approach provides a holistic view of model performance for binary classification of news article popularity.

## 🔧 Python Environment and Library Requirements

### **Python Version Used**
- **Python**: 3.13

### **Required Libraries and Minimum Versions:**

| Library | Version |
|---------|---------|
| **matplotlib** | ≥3.10.8 |
| **numpy** | ≥2.4.1 |
| **pandas** | ≥3.0.0  |
| **pyspark** | ≥4.1.1 |
| **scikit-learn** | ≥1.8.0 | 


In [ ]:
# Import required libraries
import os

from pyspark.sql import SparkSession
from pyspark.sql.functions import when, col
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.classification import LogisticRegression, NaiveBayes, DecisionTreeClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator
from pyspark.ml.tuning import TrainValidationSplit, ParamGridBuilder
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from sklearn.metrics import roc_curve, auc

# Initialize Spark Session
spark = SparkSession.builder \
    .appName("NewsPopularityClassification_3CFU") \
    .config("spark.driver.memory", "4g") \
    .getOrCreate()

print(f"Spark Version: {spark.version}")
print("Spark Session initialized successfully!")

## 1. Data Loading and Exploration

In [ ]:
# Load the dataset
data = spark.read.csv("OnlineNewsPopularity.csv", header=True, inferSchema=True)

# Clean column names (remove leading/trailing spaces)
for col_name in data.columns:
    new_col = col_name.strip()
    if new_col != col_name:
        data = data.withColumnRenamed(col_name, new_col)

print(f"Total number of articles: {data.count()}")
print(f"Number of features: {len(data.columns)}")
print("\nSchema:")
data.printSchema()

In [ ]:
# Display basic statistics
print("Sample data:")
data.show(5)

print("\nShares distribution:")
data.select("shares").describe().show()

## 2. Data Preprocessing

### 2.1 Binary Label Creation
Using threshold of 1,400 shares as per the original paper.

In [ ]:
# Create binary label: 1 if shares > 1400, 0 otherwise
data = data.withColumn("label", when(col("shares") > 1400, 1).otherwise(0))

# Check class distribution
print("Class distribution:")
data.groupBy("label").count().show()

# Calculate class percentages
total = data.count()
popular = data.filter(col("label") == 1).count()
unpopular = data.filter(col("label") == 0).count()
print(f"Popular (>1400 shares): {popular} ({popular/total*100:.2f}%)")
print(f"Unpopular (<=1400 shares): {unpopular} ({unpopular/total*100:.2f}%)")

### 2.2 Feature Selection

In [ ]:
# Define feature columns (exclude url, timedelta, shares, and label)
# timedelta is kept for sorting but excluded from features
exclude_cols = ["url", "timedelta", "shares", "label"]
feature_cols = [col for col in data.columns if col not in exclude_cols]

print(f"Number of feature columns: {len(feature_cols)}")
print(f"Feature columns: {feature_cols[:10]}...")  # Show first 10

### 2.3 Feature Vectorization

> ⚠️ **Important Note on Scaling**: Features are vectorized here but **NOT scaled yet**. 
> 
> **Scaling will be applied separately within each rolling window** to prevent data leakage by ensuring the scaler is fitted only on training data.

In [ ]:
# Assemble features into a single vector (but don't scale yet!)
assembler = VectorAssembler(inputCols=feature_cols, outputCol="features")
assembled_data = assembler.transform(data)

# Select only necessary columns (features are unscaled at this point)
# Scaling will be done separately for each rolling window to prevent data leakage
prepared_data = assembled_data.select("timedelta", "features", "label")

print("Feature assembly complete!")
print(f"Prepared data count: {prepared_data.count()}")
print("Note: Scaling will be applied separately within each rolling window to prevent data leakage")
prepared_data.show(5)

## 3. Rolling Window Time-Series Split

Following the methodology from the reference paper to ensure temporal validity:

| Parameter | Value | Description |
|-----------|-------|-------------|
| **Window size (W)** | 10,000 samples | Size of each training/testing window |
| **Step size (L)** | 1,000 samples | Number of samples to advance between windows |
| **Train/Validation split** | 70/30 | Ratio within each window |

This approach ensures that:
- ✅ **No future data leaks into past predictions**
- ✅ **Models are evaluated on truly unseen temporal data**
- ✅ **Consistent evaluation across different time periods**

In [ ]:
# Sort data by timedelta in DESCENDING order (chronological: oldest → newest)
# timedelta = days between publication and acquisition
# Large timedelta = old articles (published long ago)
# Small timedelta = new articles (published recently)
# We want: old → new (past → future) for time-series prediction
sorted_data = prepared_data.orderBy(col("timedelta").desc())

# Cache the sorted data for efficiency
sorted_data.cache()

# Parameters from the paper
W = 10000  # Window size
L = 1000   # Step size
total_rows = sorted_data.count()

print(f"Total samples: {total_rows}")
print(f"Window size (W): {W}")
print(f"Step size (L): {L}")

# Calculate number of windows
num_windows = (total_rows - W) // L + 1
print(f"Number of rolling windows: {num_windows}")

In [ ]:
print("Creating rolling windows with proper scaling and caching (this may take a few minutes)...")

# Add sequential row numbers for window extraction
# Using Window.row_number() - ensures proper ordering by timedelta
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

# Note: Window is used here to add sequential indices, not for windowed aggregations
# Using desc() order to maintain chronological order (oldest first)
window_spec = Window.orderBy(col("timedelta").desc())
data_with_row = sorted_data.withColumn("row_num", row_number().over(window_spec))
data_with_row.cache()  # Cache to avoid recomputation

# Force materialization of data_with_row
data_with_row.count()
print(f"Data with row numbers cached. Starting window creation...")

# Create windows with scaling applied within each window
windows = []
num_windows_to_create = (total_rows - W) // L + 1

for i, start in enumerate(range(0, total_rows - W + 1, L), 1):  # Using L step size as per paper
    print(f"  Creating window {i}/{num_windows_to_create}...", end=" ")
    end = start + W
    
    # Extract window data using global row numbers
    # Window contains rows (start, end] inclusive
    window_data = data_with_row.filter((col("row_num") > start) & (col("row_num") <= end))
    
    # Split into 70% train, 30% test BEFORE scaling
    # Important: split uses global row numbers for consistent boundaries
    train_size = int(W * 0.7)
    train_raw = window_data.filter(col("row_num") <= start + train_size).select("features", "label")
    test_raw = window_data.filter(col("row_num") > start + train_size).select("features", "label")
    
    # CRITICAL: Fit scaler on training data only (prevents data leakage)
    scaler = StandardScaler(inputCol="features", outputCol="scaled_features", withMean=True, withStd=True)
    scaler_model = scaler.fit(train_raw)
    
    # Transform both train and test with the scaler fitted on training data
    train_scaled = scaler_model.transform(train_raw).select(col("scaled_features").alias("features"), "label")
    test_scaled = scaler_model.transform(test_raw).select(col("scaled_features").alias("features"), "label")
    
    # OPTIMIZATION: Cache each window to avoid recomputation during training
    train_scaled.cache()
    test_scaled.cache()
    
    # Force materialization by triggering actions
    train_count = train_scaled.count()
    test_count = test_scaled.count()
    print(f"train: {train_count}, test: {test_count} - cached!")
    
    windows.append((train_scaled, test_scaled))

print(f"\n✓ Created and cached {len(windows)} rolling windows with proper scaling")
print("Each window: scaler fitted on train, applied to both train and test")
print("All windows are materialized and cached for efficient training")

## 4.  Helper Functions for Training and Evaluation

This section defines comprehensive utility functions to streamline the machine learning pipeline:

### **Evaluation Function**
- Calculates multiple performance metrics (AUC, Accuracy, F1-Score, Precision, Recall)
- Provides standardized model assessment across all algorithms

### **Training Function with Grid Search**
- Implements cross-validation within each rolling window
- Performs hyperparameter optimization via grid search
- Ensures consistent training methodology across all models

In [ ]:
# Define evaluation function
def evaluate_model(predictions):
    """
    Evaluate model predictions using multiple metrics
    
    Args:
        predictions: DataFrame with predictions
    
    Returns:
        Dictionary with evaluation metrics
    """
    # Binary classification evaluator for AUC
    binary_evaluator = BinaryClassificationEvaluator(labelCol="label", metricName="areaUnderROC")
    auc_score = binary_evaluator.evaluate(predictions)
    
    # Multiclass evaluator for other metrics
    mc_evaluator = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction")
    
    accuracy = mc_evaluator.evaluate(predictions, {mc_evaluator.metricName: "accuracy"})
    f1 = mc_evaluator.evaluate(predictions, {mc_evaluator.metricName: "f1"})
    precision = mc_evaluator.evaluate(predictions, {mc_evaluator.metricName: "weightedPrecision"})
    recall = mc_evaluator.evaluate(predictions, {mc_evaluator.metricName: "weightedRecall"})
    
    return {
        "AUC": auc_score,
        "Accuracy": accuracy,
        "F1-Score": f1,
        "Precision": precision,
        "Recall": recall
    }

def train_with_grid_search(model, param_grid, windows, model_name):
    """
    Train and evaluate model with grid search on rolling windows.
    
    Implementation follows the paper's methodology:
    1. Within each window, TrainValidationSplit performs 70/30 split for hyperparameter selection
    2. Best parameters are extracted from the validation process
    3. A NEW model with best params is trained on FULL (100%) training data
    4. Final model is tested on the 30% test set
    
    Args:
        model: PySpark ML model (estimator)
        param_grid: ParamGrid with hyperparameters to search
        windows: List of (train, test) tuples
        model_name: Name of the model for display
    
    Returns:
        Tuple: (avg_metrics_dict, all_predictions_list, final_model, best_params_list)
    """
    print(f"\n{'='*60}")
    print(f"Training {model_name} with Grid Search (Option B)")
    print(f"{'='*60}")
    print(f"Parameter grid size: {len(param_grid)} combinations")
    print(f"Step 1: TrainValidationSplit finds best params (70/30 internal split)")
    print(f"Step 2: Retrain with best params on FULL training data (100%)")
    print(f"Step 3: Test on held-out test set\n")
    
    # Create TrainValidationSplit
    # This will internally split training data into 70% train, 30% validation
    tvs = TrainValidationSplit(
        estimator=model,
        estimatorParamMaps=param_grid,
        evaluator=BinaryClassificationEvaluator(labelCol="label", metricName="areaUnderROC"),
        trainRatio=0.7,  # 70% for training, 30% for validation (as per paper)
        seed=42
    )
    
    all_metrics = []
    all_predictions = []
    best_params_per_window = []
    
    for i, (train, test) in enumerate(windows):
        print(f"Window {i+1}/{len(windows)}...")
        
        # STEP 1: Grid search finds best params using 70/30 internal split
        tvs_model = tvs.fit(train)  # Train on 70%, validate on 30%
        
        # STEP 2: Extract best parameters from the best model
        best_model_from_validation = tvs_model.bestModel
        best_params = best_model_from_validation.extractParamMap()
        
        # Store best params for analysis
        best_params_dict = {param.name: value for param, value in best_params.items()}
        best_params_per_window.append(best_params_dict)
        
        # STEP 3: CRITICAL - Retrain with best params on FULL training data (100%)
        # Create new model instance and copy best params
        final_model = model.copy(best_params)
        
        # Train on full training data of this window (not just 70%!)
        final_model = final_model.fit(train)
        
        # STEP 4: Now predict on test set using model trained on full data
        predictions = final_model.transform(test)
        
        # Evaluate
        metrics = evaluate_model(predictions)
        all_metrics.append(metrics)
        
        # Store predictions for ROC curve
        pred_df = predictions.select("label", "probability").toPandas()
        all_predictions.append(pred_df)
        
        print(f"  → Best params (sample): {list(best_params.items())[:3]}")
        print(f"  → Test AUC: {metrics['AUC']:.4f}, Accuracy: {metrics['Accuracy']:.4f}\n")
    
    # Calculate average metrics
    avg_metrics = {}
    for metric_name in all_metrics[0].keys():
        avg_metrics[metric_name] = np.mean([m[metric_name] for m in all_metrics])
    
    print(f"\n{model_name} - Average Metrics across {len(windows)} windows:")
    print(f"  AUC:       {avg_metrics['AUC']:.4f}")
    print(f"  Accuracy:  {avg_metrics['Accuracy']:.4f}")
    print(f"  F1-Score:  {avg_metrics['F1-Score']:.4f}")
    print(f"  Precision: {avg_metrics['Precision']:.4f}")
    print(f"  Recall:    {avg_metrics['Recall']:.4f}")
    
    # Analyze best parameters across windows
    print(f"\n{model_name} - Hyperparameter Analysis:")
    print(f"  Best parameters found across {len(windows)} windows")
    print(f"  (Showing first 5 windows as examples)")
    for i, params in enumerate(best_params_per_window[:5]):
        print(f"  Window {i+1}: {params}")
    
    return avg_metrics, all_predictions, final_model, best_params_per_window

print("Helper functions defined successfully!")

In [ ]:
from pyspark.ml.classification import RandomForestClassifier, GBTClassifier, LinearSVC 

### Random Forest Classifier

**Random Forest** creates multiple decision trees and aggregates their predictions for improved accuracy and reduced overfitting.

In [ ]:
# Initialize Random Forest model
rf = RandomForestClassifier(
    featuresCol="features",
    labelCol="label"
)

# Define parameter grid for Random Forest
# Based on common hyperparameters for ensemble methods
rf_param_grid = ParamGridBuilder() \
    .addGrid(rf.numTrees, [50, 100, 150]) \
    .addGrid(rf.maxDepth, [5, 10, 15]) \
    .addGrid(rf.minInfoGain, [0.0, 0.01]) \
    .build()

# Train and evaluate with grid search
rf_metrics, rf_predictions, rf_model, rf_best_params = train_with_grid_search(
    rf, rf_param_grid, windows, "Random Forest"
)

26/01/30 14:13:08 WARN DAGScheduler: Broadcasting large task binary with size 13.8 MiB
26/01/30 14:13:10 WARN DAGScheduler: Broadcasting large task binary with size 1191.2 KiB
26/01/30 14:13:14 WARN DAGScheduler: Broadcasting large task binary with size 16.7 MiB


### Gradient-Boosted Tree Classifier

**Gradient-Boosted Trees** build an ensemble of weak learners sequentially, where each new tree corrects errors made by previous trees.


In [ ]:
# Initialize Gradient-Boosted Tree model
gbt = GBTClassifier(
    featuresCol="features",
    labelCol="label"
)

# Define parameter grid for Gradient-Boosted Tree
# Based on common hyperparameters for boosting methods
gbt_param_grid = ParamGridBuilder() \
    .addGrid(gbt.maxIter, [20, 50, 100]) \
    .addGrid(gbt.maxDepth, [3, 5, 7]) \
    .addGrid(gbt.stepSize, [0.05, 0.1, 0.2]) \
    .build()

# Train and evaluate with grid search
gbt_metrics, gbt_predictions, gbt_model, gbt_best_params = train_with_grid_search(
    gbt, gbt_param_grid, windows, "Gradient-Boosted Tree"
)

### Linear SVM (Support Vector Machine)


In [ ]:
# Initialize Linear SVM model
svm = LinearSVC(
    featuresCol="features",
    labelCol="label"
)

# Define parameter grid for Linear SVM
# Based on common hyperparameters for SVM
svm_param_grid = ParamGridBuilder() \
    .addGrid(svm.regParam, [0.001, 0.01, 0.1, 1.0]) \
    .addGrid(svm.maxIter, [100, 200, 500]) \
    .build()


# Train and evaluate with grid search
svm_metrics, svm_predictions, svm_model, svm_best_params = train_with_grid_search(
    svm, svm_param_grid, windows, "Linear SVM"
)

### Model Comparison

Performance comparison to identify the best-performing approach for online news popularity prediction.

In [ ]:
# Comparison of 3 algorithms
extended_comparison_df = pd.DataFrame({
    'Model': [
        'Logistic Regression', 'Naive Bayes', 'Decision Tree',
        'Random Forest', 'Gradient-Boosted Tree', 'Linear SVM'
    ],
    'AUC': [
        rf_metrics['AUC'], gbt_metrics['AUC'], svm_metrics['AUC']
    ],
    'Accuracy': [
        rf_metrics['Accuracy'], gbt_metrics['Accuracy'], svm_metrics['Accuracy']
    ],
    'F1-Score': [
        rf_metrics['F1-Score'], gbt_metrics['F1-Score'], svm_metrics['F1-Score']
    ],
    'Precision': [
        rf_metrics['Precision'], gbt_metrics['Precision'], svm_metrics['Precision']
    ],
    'Recall': [
        rf_metrics['Recall'], gbt_metrics['Recall'], svm_metrics['Recall']
    ]
})

print("\n" + "="*100)
print("MODEL COMPARISON ")
print("="*100)
print(extended_comparison_df.to_string(index=False))

# Find best performing model for each metric
print(f"\nBest AUC: {extended_comparison_df.loc[extended_comparison_df['AUC'].idxmax(), 'Model']} ({extended_comparison_df['AUC'].max():.4f})")
print(f"Best Accuracy: {extended_comparison_df.loc[extended_comparison_df['Accuracy'].idxmax(), 'Model']} ({extended_comparison_df['Accuracy'].max():.4f})")
print(f"Best F1-Score: {extended_comparison_df.loc[extended_comparison_df['F1-Score'].idxmax(), 'Model']} ({extended_comparison_df['F1-Score'].max():.4f})")
print(f"Best Precision: {extended_comparison_df.loc[extended_comparison_df['Precision'].idxmax(), 'Model']} ({extended_comparison_df['Precision'].max():.4f})")
print(f"Best Recall: {extended_comparison_df.loc[extended_comparison_df['Recall'].idxmax(), 'Model']} ({extended_comparison_df['Recall'].max():.4f})")

# Visualize extended comparison
fig, axes = plt.subplots(2, 2, figsize=(18, 12))

# 1. Bar chart for all metrics
metrics_to_plot = ['AUC', 'Accuracy', 'F1-Score', 'Precision', 'Recall']
x = np.arange(len(metrics_to_plot))
width = 0.13
colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b']

for i, (model, color) in enumerate(zip(extended_comparison_df['Model'], colors)):
    values = [extended_comparison_df.loc[extended_comparison_df['Model'] == model, metric].values[0] for metric in metrics_to_plot]
    axes[0,0].bar(x + i*width, values, width, label=model, color=color)

axes[0,0].set_xlabel('Metrics')
axes[0,0].set_ylabel('Score')
axes[0,0].set_title('Extended Model Performance Comparison')
axes[0,0].set_xticks(x + width * 2.5)
axes[0,0].set_xticklabels(metrics_to_plot, rotation=45)
axes[0,0].legend(bbox_to_anchor=(1.05, 1), loc='upper left')
axes[0,0].grid(axis='y', alpha=0.3)

# 2. AUC comparison
models = extended_comparison_df['Model']
aucs = extended_comparison_df['AUC']
bars = axes[0,1].barh(models, aucs, color=colors)
axes[0,1].set_xlabel('AUC Score')
axes[0,1].set_title('AUC Comparison - All Models')
axes[0,1].grid(axis='x', alpha=0.3)
# Add value labels on bars
for bar, auc in zip(bars, aucs):
    axes[0,1].text(bar.get_width() + 0.01, bar.get_y() + bar.get_height()/2, 
                  f'{auc:.3f}', ha='left', va='center', fontsize=10)

# 3. Accuracy vs F1-Score scatter plot
axes[1,0].scatter(extended_comparison_df['Accuracy'], extended_comparison_df['F1-Score'], 
                 c=colors, s=100, alpha=0.7)
for i, model in enumerate(extended_comparison_df['Model']):
    axes[1,0].annotate(model, 
                      (extended_comparison_df['Accuracy'].iloc[i], extended_comparison_df['F1-Score'].iloc[i]),
                      xytext=(5, 5), textcoords='offset points', fontsize=9)
axes[1,0].set_xlabel('Accuracy')
axes[1,0].set_ylabel('F1-Score')
axes[1,0].set_title('Accuracy vs F1-Score')
axes[1,0].grid(alpha=0.3)

# 4. Precision vs Recall scatter plot
axes[1,1].scatter(extended_comparison_df['Recall'], extended_comparison_df['Precision'], 
                 c=colors, s=100, alpha=0.7)
for i, model in enumerate(extended_comparison_df['Model']):
    axes[1,1].annotate(model, 
                      (extended_comparison_df['Recall'].iloc[i], extended_comparison_df['Precision'].iloc[i]),
                      xytext=(5, 5), textcoords='offset points', fontsize=9)
axes[1,1].set_xlabel('Recall')
axes[1,1].set_ylabel('Precision')
axes[1,1].set_title('Precision vs Recall')
axes[1,1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

### ROC Curves 

In [ ]:
# Plot ROC curves for new classifiers
plt.figure(figsize=(12, 8))

# New models data
new_models_data = [
    ('Random Forest', rf_predictions, '#d62728'),
    ('Gradient-Boosted Tree', gbt_predictions, '#9467bd'), 
    ('Linear SVM', svm_predictions, '#8c564b')
]

for model_name, predictions_list, color in new_models_data:
    # Combine all windows
    all_labels = []
    all_probs = []
    
    for pred_df in predictions_list:
        all_labels.extend(pred_df['label'].values)
        if 'probability' in pred_df.columns:
            # For models with probability output (RF, GBT)
            probs = [p[1] if len(p) > 1 else p[0] for p in pred_df['probability'].values]
        else:
            # For Linear SVM (no probability, use prediction/rawPrediction)
            # We'll use a simple approach for SVM ROC
            probs = pred_df.get('prediction', pred_df.get('rawPrediction', [0.5] * len(pred_df))).values
            if hasattr(probs[0], '__len__') and len(probs[0]) > 1:
                probs = [p[1] for p in probs]
        all_probs.extend(probs)
    
    # Calculate ROC curve
    fpr, tpr, _ = roc_curve(all_labels, all_probs)
    roc_auc = auc(fpr, tpr)
    
    # Plot
    plt.plot(fpr, tpr, color=color, lw=2, 
             label=f'{model_name} (AUC = {roc_auc:.4f})')

# Plot diagonal line
plt.plot([0, 1], [0, 1], 'k--', lw=2, label='Random Classifier')

plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curves - New Classifiers')
plt.legend(loc="lower right")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Comprehensive ROC curve comparison
plt.figure(figsize=(14, 10))

# All models data
all_models_data = [
    ('Random Forest', rf_predictions, '#d62728'),
    ('Gradient-Boosted Tree', gbt_predictions, '#9467bd'),
    ('Linear SVM', svm_predictions, '#8c564b')
]

auc_scores = []

for model_name, predictions_list, color in all_models_data:
    # Combine all windows
    all_labels = []
    all_probs = []
    
    for pred_df in predictions_list:
        all_labels.extend(pred_df['label'].values)
        if 'probability' in pred_df.columns:
            # For models with probability output
            probs = [p[1] if len(p) > 1 else p[0] for p in pred_df['probability'].values]
        else:
            # For Linear SVM (no probability output)
            probs = pred_df.get('prediction', pred_df.get('rawPrediction', [0.5] * len(pred_df))).values
            if hasattr(probs[0], '__len__') and len(probs[0]) > 1:
                probs = [p[1] for p in probs]
        all_probs.extend(probs)
    
    # Calculate ROC curve
    fpr, tpr, _ = roc_curve(all_labels, all_probs)
    roc_auc = auc(fpr, tpr)
    auc_scores.append((model_name, roc_auc))
    
    # Plot
    plt.plot(fpr, tpr, color=color, lw=2.5, 
             label=f'{model_name} (AUC = {roc_auc:.4f})')

# Plot diagonal line
plt.plot([0, 1], [0, 1], 'k--', lw=1, alpha=0.5, label='Random Classifier')

plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate', fontsize=12)
plt.ylabel('True Positive Rate', fontsize=12)
plt.title('Comprehensive ROC Curves', fontsize=14, fontweight='bold')
plt.legend(loc="lower right", fontsize=11)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

# Summary of all AUC scores
print("\n" + "="*60)
print("COMPREHENSIVE AUC SUMMARY")
print("="*60)

# Sort by AUC score (descending)
auc_scores.sort(key=lambda x: x[1], reverse=True)

for i, (model, auc_score) in enumerate(auc_scores):
    print(f"{i+1}. {model:<25} AUC = {auc_score:.4f}")

print(f"\nBest performing model: {auc_scores[0][0]} (AUC = {auc_scores[0][1]:.4f})")
print(f"AUC improvement over baseline: {auc_scores[0][1] - 0.5:.4f}")

In [ ]:
# Final Summary of the Big Data Analysis Project


extended_models = ["Random Forest", "Gradient-Boosted Tree", "Linear SVM"]


print("\n Models:")
for i, model in enumerate(extended_models, 4):
    print(f"  {i}. {model}")

# Display final rankings based on AUC
print(f"\n MODEL PERFORMANCE RANKING (by AUC):")
for i, (model, auc_score) in enumerate(auc_scores, 1):
    star = " ⭐" if i == 1 else ""
    print(f"  {i}. {model:<25} AUC = {auc_score:.4f}{star}")

print(f"\n KEY FINDINGS:")
print(f"• Best performing model: {auc_scores[0][0]}")
print(f"• Highest AUC achieved: {auc_scores[0][1]:.4f}")
print(f"• Average AUC across all models: {np.mean([score[1] for score in auc_scores]):.4f}")
